
# STEP 6 — LEAN Practical Consequence

## Goal

Test the practical systems implication of Residual Approximation Complexity without running a huge grid.

We ask:

> For a target absolute error, can a **cheaper baseline + learned corrector** match or beat a much more expensive baseline-only system?

Core quantities:

\[
D(B)=\mathbb E\|\Phi-B\|^2
\]

\[
E_B(K)=
\frac{\mathbb E\|(\Phi-B)-G_K\|^2}
{\mathbb E\|\Phi-B\|^2}
\]

and therefore

\[
J(B,K)=D(B)E_B(K),
\]

the absolute corrected error.

## Lean design

Baseline-only errors are measured for

\[
C\in\{10,20,50,100,200,500\}.
\]

Correctors are trained only for the practically informative subset

\[
C\in\{50,100,200,500\},
\]

with widths

\[
w\in\{1,8,32\},
\]

and 3 seeds.

That is only

\[
4\times3\times3=36
\]

training runs instead of 108.

The notebook also reuses any matching runs already completed by the larger Step 6 notebook.


In [1]:

from pathlib import Path
import copy
import math
import gc
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED_BASE = 20260926
random.seed(SEED_BASE)
np.random.seed(SEED_BASE)
torch.manual_seed(SEED_BASE)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 2050


## 1. Load existing CIFAR tensors

In [2]:

DATA_DIR = Path("ddim_residual_data_512")
OUT_DIR = Path("step6_joint_frontier_lean")
OUT_DIR.mkdir(exist_ok=True)

ALL_C = [10, 20, 50, 100, 200, 500]

# Only these receive correction models in the lean experiment
CORRECTED_C = [50, 100, 200, 500]

WIDTHS = [1, 8, 32]
SEEDS = [0, 1, 2]

required = [
    DATA_DIR / "z.pt",
    DATA_DIR / "teacher_1000.pt",
] + [
    DATA_DIR / f"baseline_{C}.pt"
    for C in ALL_C
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required file: {p.resolve()}"
        )

z_all = torch.load(
    DATA_DIR / "z.pt",
    map_location="cpu"
)

teacher = torch.load(
    DATA_DIR / "teacher_1000.pt",
    map_location="cpu"
)

baselines = {
    C: torch.load(
        DATA_DIR / f"baseline_{C}.pt",
        map_location="cpu"
    )
    for C in ALL_C
}

assert len(z_all) == 512
assert len(teacher) == 512

for C in ALL_C:
    assert baselines[C].shape == teacher.shape
    assert torch.isfinite(baselines[C]).all()

assert torch.isfinite(z_all).all()
assert torch.isfinite(teacher).all()

print("Loaded:", tuple(z_all.shape))
print("Teacher:", tuple(teacher.shape))
print("All baselines:", ALL_C)
print("Corrected baselines:", CORRECTED_C)
print("Widths:", WIDTHS)
print("Seeds:", SEEDS)
print("Total lean training runs:", len(CORRECTED_C)*len(WIDTHS)*len(SEEDS))


Loaded: (512, 3, 32, 32)
Teacher: (512, 3, 32, 32)
All baselines: [10, 20, 50, 100, 200, 500]
Corrected baselines: [50, 100, 200, 500]
Widths: [1, 8, 32]
Seeds: [0, 1, 2]
Total lean training runs: 36


## 2. Fixed split + baseline-only error

In [3]:

TRAIN_END = 384
VAL_END = 448

z_train = z_all[:TRAIN_END]
z_val = z_all[TRAIN_END:VAL_END]
z_test = z_all[VAL_END:]

residuals = {}
baseline_rows = []

for C in ALL_C:
    R = teacher - baselines[C]
    residuals[C] = R

    test_mse = (
        R[VAL_END:] ** 2
    ).mean().item()

    baseline_rows.append({
        "C": C,
        "test_residual_mse": test_mse,
        "test_residual_rms": test_mse ** 0.5,
    })

baseline_df = (
    pd.DataFrame(baseline_rows)
    .sort_values("C")
    .reset_index(drop=True)
)

display(baseline_df)

baseline_df.to_csv(
    OUT_DIR / "baseline_test_error.csv",
    index=False
)


,C,test_residual_mse,test_residual_rms
0,10,0.199645,0.446817
1,20,0.206125,0.454010
2,50,0.198522,0.445558
3,100,0.180091,0.424372
4,200,0.144920,0.380684
5,500,0.070779,0.266043


## 3. Corrector + trainer

In [4]:

class CorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def train_corrector(
    z_train,
    r_train,
    z_val,
    r_val,
    z_test,
    r_test,
    width,
    seed,
    max_epochs=700,
    patience=50,
    batch_size=32,
    lr=3e-4,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = CorrectionCNN(width).to(device)

    train_energy = (r_train ** 2).mean().item()
    train_rms = train_energy ** 0.5

    y_train = r_train / train_rms

    loader = DataLoader(
        TensorDataset(z_train, y_train),
        batch_size=batch_size,
        shuffle=True,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    loss_fn = nn.MSELoss()

    val_energy = (r_val ** 2).mean().item()

    model.eval()
    with torch.no_grad():
        pred_val = (
            model(z_val.to(device)).cpu()
            * train_rms
        )

    best_val_nmse = (
        ((pred_val-r_val)**2).mean().item()
        / val_energy
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    best_epoch = 0
    bad_epochs = 0

    for epoch in range(max_epochs):
        model.train()

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite training loss."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

        model.eval()

        with torch.no_grad():
            pred_val = (
                model(z_val.to(device)).cpu()
                * train_rms
            )

        val_nmse = (
            ((pred_val-r_val)**2).mean().item()
            / val_energy
        )

        if val_nmse < best_val_nmse - 1e-5:
            best_val_nmse = val_nmse
            best_state = copy.deepcopy(
                model.state_dict()
            )
            best_epoch = epoch + 1
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_test = (
            model(z_test.to(device)).cpu()
            * train_rms
        )

    test_energy = (
        r_test ** 2
    ).mean().item()

    test_mse = (
        (pred_test-r_test)**2
    ).mean().item()

    test_nmse = (
        test_mse / test_energy
    )

    result = {
        "width": width,
        "params": count_params(model),
        "seed": seed,
        "best_epoch": best_epoch,
        "val_nmse": best_val_nmse,
        "test_nmse": test_nmse,
        "test_corrected_mse": test_mse,
    }

    del model
    gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()

    return result


for w in WIDTHS:
    m = CorrectionCNN(w)
    print(
        f"width={w:2d}",
        f"params={count_params(m):6d}"
    )


width= 1 params=    78
width= 8 params=  1611
width=32 params= 20259



## 4. Reuse partial Step 6 results if they already exist

If the larger Step 6 notebook already completed some matching runs, we import them and skip retraining.


In [5]:

LEAN_RESULTS_PATH = (
    OUT_DIR / "joint_grid_results_lean.csv"
)

HEAVY_RESULTS_PATH = (
    Path("step6_joint_frontier")
    / "joint_grid_results.csv"
)

results = []

# Load already-completed lean rows
if LEAN_RESULTS_PATH.exists():
    lean_old = pd.read_csv(
        LEAN_RESULTS_PATH
    )

    results.extend(
        lean_old.to_dict("records")
    )

    print(
        "Loaded lean rows:",
        len(lean_old)
    )

# Reuse matching heavy rows
if HEAVY_RESULTS_PATH.exists():
    heavy = pd.read_csv(
        HEAVY_RESULTS_PATH
    )

    wanted = heavy[
        heavy["C"].isin(CORRECTED_C)
        & heavy["width"].isin(WIDTHS)
        & heavy["seed"].isin(SEEDS)
    ].copy()

    existing_keys = {
        (
            int(r["C"]),
            int(r["width"]),
            int(r["seed"]),
        )
        for r in results
    }

    reused = 0

    for _, row in wanted.iterrows():
        key = (
            int(row["C"]),
            int(row["width"]),
            int(row["seed"]),
        )

        if key not in existing_keys:
            results.append(
                row.to_dict()
            )
            existing_keys.add(key)
            reused += 1

    print(
        "Reused matching heavy rows:",
        reused
    )

pd.DataFrame(results).to_csv(
    LEAN_RESULTS_PATH,
    index=False
)

print(
    "Rows already available:",
    len(results)
)


Loaded lean rows: 36
Reused matching heavy rows: 0
Rows already available: 36


## 5. Run only missing lean combinations

In [6]:

def already_done(C, width, seed):
    return any(
        int(r["C"]) == C
        and int(r["width"]) == width
        and int(r["seed"]) == seed
        for r in results
    )


for C in CORRECTED_C:
    R = residuals[C]

    r_train = R[:TRAIN_END]
    r_val = R[TRAIN_END:VAL_END]
    r_test = R[VAL_END:]

    baseline_test_mse = (
        r_test ** 2
    ).mean().item()

    for width in WIDTHS:
        for seed in SEEDS:

            if already_done(
                C, width, seed
            ):
                print(
                    "SKIP",
                    "C=", C,
                    "width=", width,
                    "seed=", seed
                )
                continue

            print(
                "\nRUN",
                "C=", C,
                "width=", width,
                "seed=", seed
            )

            out = train_corrector(
                z_train,
                r_train,
                z_val,
                r_val,
                z_test,
                r_test,
                width=width,
                seed=seed,
            )

            out["C"] = C
            out[
                "baseline_test_mse"
            ] = baseline_test_mse

            results.append(out)

            pd.DataFrame(
                results
            ).to_csv(
                LEAN_RESULTS_PATH,
                index=False
            )

joint_df = (
    pd.DataFrame(results)
    .drop_duplicates(
        subset=["C","width","seed"],
        keep="last"
    )
    .sort_values(
        ["C","width","seed"]
    )
    .reset_index(drop=True)
)

print(
    "\nCompleted rows:",
    len(joint_df),
    "/",
    len(CORRECTED_C)
    * len(WIDTHS)
    * len(SEEDS)
)

display(joint_df.head())


SKIP C= 50 width= 1 seed= 0
SKIP C= 50 width= 1 seed= 1
SKIP C= 50 width= 1 seed= 2
SKIP C= 50 width= 8 seed= 0
SKIP C= 50 width= 8 seed= 1
SKIP C= 50 width= 8 seed= 2
SKIP C= 50 width= 32 seed= 0
SKIP C= 50 width= 32 seed= 1
SKIP C= 50 width= 32 seed= 2
SKIP C= 100 width= 1 seed= 0
SKIP C= 100 width= 1 seed= 1
SKIP C= 100 width= 1 seed= 2
SKIP C= 100 width= 8 seed= 0
SKIP C= 100 width= 8 seed= 1
SKIP C= 100 width= 8 seed= 2
SKIP C= 100 width= 32 seed= 0
SKIP C= 100 width= 32 seed= 1
SKIP C= 100 width= 32 seed= 2
SKIP C= 200 width= 1 seed= 0
SKIP C= 200 width= 1 seed= 1
SKIP C= 200 width= 1 seed= 2
SKIP C= 200 width= 8 seed= 0
SKIP C= 200 width= 8 seed= 1
SKIP C= 200 width= 8 seed= 2
SKIP C= 200 width= 32 seed= 0
SKIP C= 200 width= 32 seed= 1
SKIP C= 200 width= 32 seed= 2
SKIP C= 500 width= 1 seed= 0
SKIP C= 500 width= 1 seed= 1
SKIP C= 500 width= 1 seed= 2
SKIP C= 500 width= 8 seed= 0
SKIP C= 500 width= 8 seed= 1
SKIP C= 500 width= 8 seed= 2
SKIP C= 500 width= 32 seed= 0
SKIP C= 500 w

,width,params,seed,best_epoch,val_nmse,test_nmse,test_corrected_mse,C,baseline_test_mse
0,1,78,0,33,0.357079,0.360649,0.071597,50,0.198522
1,1,78,1,699,0.336599,0.338134,0.067127,50,0.198522
2,1,78,2,688,0.330864,0.334695,0.066444,50,0.198522
3,8,1611,0,501,0.316434,0.321364,0.063798,50,0.198522
4,8,1611,1,530,0.317135,0.320487,0.063624,50,0.198522


## 6. Joint corrected-error summary

In [7]:

joint_summary = (
    joint_df
    .groupby(
        ["C","width","params"],
        as_index=False
    )
    .agg(
        baseline_test_mse=(
            "baseline_test_mse",
            "mean"
        ),
        nmse_mean=(
            "test_nmse",
            "mean"
        ),
        nmse_std=(
            "test_nmse",
            "std"
        ),
        corrected_mse_mean=(
            "test_corrected_mse",
            "mean"
        ),
        corrected_mse_std=(
            "test_corrected_mse",
            "std"
        ),
    )
)

display(joint_summary)

joint_summary.to_csv(
    OUT_DIR / "joint_grid_summary.csv",
    index=False
)


,C,width,params,baseline_test_mse,nmse_mean,nmse_std,corrected_mse_mean,corrected_mse_std
0,50,1,78,0.198522,0.344493,0.014097,0.068389,0.002799
1,50,8,1611,0.198522,0.321945,0.001820,0.063913,0.000361
2,50,32,20259,0.198522,0.318621,0.000970,0.063253,0.000192
3,100,1,78,0.180091,0.359565,0.014101,0.064755,0.002540
4,100,8,1611,0.180091,0.336396,0.001497,0.060582,0.000270
5,100,32,20259,0.180091,0.333112,0.001184,0.059991,0.000213
6,200,1,78,0.144920,0.392885,0.013831,0.056937,0.002004
7,200,8,1611,0.144920,0.369615,0.001243,0.053565,0.000180
8,200,32,20259,0.144920,0.366853,0.000890,0.053164,0.000129
9,500,1,78,0.070779,0.562505,0.003121,0.039814,0.000221


## 7. Best corrected design at each baseline compute

In [8]:

best_corrected_by_C = (
    joint_summary
    .sort_values(
        ["C","corrected_mse_mean"]
    )
    .groupby("C", as_index=False)
    .first()
)

display(
    best_corrected_by_C[
        [
            "C",
            "baseline_test_mse",
            "width",
            "params",
            "nmse_mean",
            "corrected_mse_mean",
            "corrected_mse_std",
        ]
    ]
)

best_corrected_by_C.to_csv(
    OUT_DIR / "best_corrected_by_C.csv",
    index=False
)


,C,baseline_test_mse,width,params,nmse_mean,corrected_mse_mean,corrected_mse_std
0,50,0.198522,32,20259,0.318621,0.063253,0.000192
1,100,0.180091,32,20259,0.333112,0.059991,0.000213
2,200,0.144920,32,20259,0.366853,0.053164,0.000129
3,500,0.070779,32,20259,0.550977,0.038998,0.000084



## 8. Target-error decision table

For each target error \(\varepsilon\):

**Baseline-only policy**

\[
\min C \quad \text{s.t.}\quad D(B_C)\le\varepsilon.
\]

**Joint policy**

Among measured corrected designs, select the smallest baseline NFE \(C\) that reaches the target, then the smallest corrector parameter count.


In [9]:

# Build targets from the observed baseline range plus
# several practical hand-picked values.
TARGET_EPS = sorted(set([
    0.20,
    0.15,
    0.12,
    0.10,
    0.09,
    0.08,
    0.07,
    0.06,
]))

decision_rows = []

for eps in TARGET_EPS:

    feasible_base = (
        baseline_df[
            baseline_df[
                "test_residual_mse"
            ] <= eps
        ]
        .sort_values("C")
    )

    if len(feasible_base):
        rb = feasible_base.iloc[0]
        base_C = int(rb["C"])
        base_err = float(
            rb["test_residual_mse"]
        )
    else:
        base_C = np.nan
        base_err = np.nan

    feasible_joint = (
        joint_summary[
            joint_summary[
                "corrected_mse_mean"
            ] <= eps
        ]
        .sort_values(
            ["C","params"]
        )
    )

    if len(feasible_joint):
        rj = feasible_joint.iloc[0]

        joint_C = int(rj["C"])
        joint_width = int(rj["width"])
        joint_params = int(
            rj["params"]
        )
        joint_err = float(
            rj["corrected_mse_mean"]
        )
    else:
        joint_C = np.nan
        joint_width = np.nan
        joint_params = np.nan
        joint_err = np.nan

    decision_rows.append({
        "target_epsilon": eps,
        "baseline_only_min_C":
            base_C,
        "baseline_only_error":
            base_err,
        "joint_min_C":
            joint_C,
        "joint_width":
            joint_width,
        "joint_params":
            joint_params,
        "joint_error":
            joint_err,
        "NFE_saved":
            (
                base_C-joint_C
                if (
                    not pd.isna(base_C)
                    and not pd.isna(joint_C)
                )
                else np.nan
            ),
        "joint_over_baseline_NFE":
            (
                joint_C/base_C
                if (
                    not pd.isna(base_C)
                    and not pd.isna(joint_C)
                )
                else np.nan
            ),
    })

decision_df = pd.DataFrame(
    decision_rows
)

display(decision_df)

decision_df.to_csv(
    OUT_DIR / "target_error_decisions.csv",
    index=False
)


,target_epsilon,baseline_only_min_C,baseline_only_error,joint_min_C,joint_width,joint_params,joint_error,NFE_saved,joint_over_baseline_NFE
0,0.06,NaN,NaN,100,32,20259,0.059991,NaN,NaN
1,0.07,NaN,NaN,50,1,78,0.068389,NaN,NaN
2,0.08,500.0,0.070779,50,1,78,0.068389,450.0,0.10
3,0.09,500.0,0.070779,50,1,78,0.068389,450.0,0.10
4,0.10,500.0,0.070779,50,1,78,0.068389,450.0,0.10
5,0.12,500.0,0.070779,50,1,78,0.068389,450.0,0.10
6,0.15,200.0,0.144920,50,1,78,0.068389,150.0,0.25
7,0.20,10.0,0.199645,50,1,78,0.068389,-40.0,5.00



## 9. Lean latency benchmark

We only benchmark the four corrected baselines and three corrector widths.

This is secondary evidence; NFE remains the clean algorithmic compute metric.


In [10]:

RUN_LATENCY = True
latency_available = False


In [11]:

if RUN_LATENCY:
    try:
        from diffusers import (
            DDPMPipeline,
            DDIMScheduler
        )

        MODEL_ID = "google/ddpm-cifar10-32"

        pipe = (
            DDPMPipeline
            .from_pretrained(MODEL_ID)
        )

        unet = (
            pipe.unet
            .to(device)
            .eval()
        )

        scheduler_config = (
            pipe.scheduler.config
        )

        bench_z = z_test[:4].to(device)

        def sync():
            if device == "cuda":
                torch.cuda.synchronize()

        @torch.no_grad()
        def run_sampler(C):
            scheduler = (
                DDIMScheduler
                .from_config(
                    scheduler_config
                )
            )

            scheduler.set_timesteps(
                C,
                device=device
            )

            x = bench_z.clone()

            for t in scheduler.timesteps:
                model_input = (
                    scheduler
                    .scale_model_input(
                        x,t
                    )
                )

                pred = (
                    unet(
                        model_input,
                        t
                    ).sample
                )

                x = scheduler.step(
                    pred,
                    t,
                    x,
                    eta=0.0
                ).prev_sample

            return x

        # Warm-up
        _ = run_sampler(10)
        sync()

        sampler_rows = []

        for C in CORRECTED_C:
            sync()
            t0 = time.perf_counter()

            _ = run_sampler(C)

            sync()

            ms = (
                1000
                * (
                    time.perf_counter()-t0
                )
                / len(bench_z)
            )

            sampler_rows.append({
                "C": C,
                "sampler_ms_per_image":
                    ms
            })

        sampler_latency_df = (
            pd.DataFrame(
                sampler_rows
            )
        )

        corr_rows = []

        for width in WIDTHS:
            model = (
                CorrectionCNN(width)
                .to(device)
                .eval()
            )

            # warm-up
            with torch.no_grad():
                for _ in range(10):
                    _ = model(bench_z)

            sync()

            times = []

            with torch.no_grad():
                for _ in range(30):
                    sync()
                    t0 = time.perf_counter()

                    _ = model(bench_z)

                    sync()

                    times.append(
                        1000
                        * (
                            time.perf_counter()-t0
                        )
                        / len(bench_z)
                    )

            corr_rows.append({
                "width": width,
                "params":
                    count_params(model),
                "corrector_ms_per_image":
                    float(
                        np.median(times)
                    ),
            })

            del model

        corrector_latency_df = (
            pd.DataFrame(
                corr_rows
            )
        )

        display(sampler_latency_df)
        display(corrector_latency_df)

        sampler_latency_df.to_csv(
            OUT_DIR /
            "sampler_latency.csv",
            index=False
        )

        corrector_latency_df.to_csv(
            OUT_DIR /
            "corrector_latency.csv",
            index=False
        )

        latency_available = True

    except Exception as e:
        print(
            "Latency section skipped:"
        )
        print(repr(e))
        latency_available = False


C:\Users\Public\Anaconda3\envs\tabpfn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Loading pipeline components...: 100%|██████████| 2/2 [00:00<00:00, 12.09it/s]


,C,sampler_ms_per_image
0,50,501.905250
1,100,933.239175
2,200,1857.077750
3,500,4670.901250


,width,params,corrector_ms_per_image
0,1,78,0.131813
1,8,1611,0.131112
2,32,20259,0.148562


## 10. Final lean Step 6 summary

In [12]:

checks = {}

valid = decision_df.dropna(
    subset=[
        "baseline_only_min_C",
        "joint_min_C"
    ]
)

checks[
    "joint_policy_saves_NFE_somewhere"
] = bool(
    len(valid)
    and (
        valid["joint_min_C"]
        <
        valid["baseline_only_min_C"]
    ).any()
)

checks[
    "joint_policy_saves_2x_or_more_NFE_somewhere"
] = bool(
    len(valid)
    and (
        valid[
            "joint_over_baseline_NFE"
        ] <= 0.5
    ).any()
)

# Does the practical corrected frontier flatten?
lookup = (
    best_corrected_by_C
    .set_index("C")
)

if (
    100 in lookup.index
    and 500 in lookup.index
):
    err100 = float(
        lookup.loc[
            100,
            "corrected_mse_mean"
        ]
    )

    err500 = float(
        lookup.loc[
            500,
            "corrected_mse_mean"
        ]
    )

    checks[
        "C100_corrected_within_15pct_of_C500_corrected"
    ] = bool(
        err100 <= 1.15*err500
    )
else:
    checks[
        "C100_corrected_within_15pct_of_C500_corrected"
    ] = False


print("="*80)
print("LEAN STEP 6 SUMMARY")
print("="*80)

print("\nBASELINE ERRORS")
display(baseline_df)

print(
    "\nBEST CORRECTED DESIGN "
    "AT EACH C"
)
display(best_corrected_by_C)

print(
    "\nTARGET-ERROR DECISIONS"
)
display(decision_df)

print("\nAUTOMATIC CHECKS")

for k,v in checks.items():
    print(f"{k}: {v}")

print(
    "\nLatency available:",
    latency_available
)

pd.DataFrame(
    [checks]
).to_csv(
    OUT_DIR /
    "automatic_checks.csv",
    index=False
)

print(
    "\nSaved outputs to:",
    OUT_DIR.resolve()
)


LEAN STEP 6 SUMMARY

BASELINE ERRORS


,C,test_residual_mse,test_residual_rms
0,10,0.199645,0.446817
1,20,0.206125,0.454010
2,50,0.198522,0.445558
3,100,0.180091,0.424372
4,200,0.144920,0.380684
5,500,0.070779,0.266043



BEST CORRECTED DESIGN AT EACH C


,C,width,params,baseline_test_mse,nmse_mean,nmse_std,corrected_mse_mean,corrected_mse_std
0,50,32,20259,0.198522,0.318621,0.000970,0.063253,0.000192
1,100,32,20259,0.180091,0.333112,0.001184,0.059991,0.000213
2,200,32,20259,0.144920,0.366853,0.000890,0.053164,0.000129
3,500,32,20259,0.070779,0.550977,0.001188,0.038998,0.000084



TARGET-ERROR DECISIONS


,target_epsilon,baseline_only_min_C,baseline_only_error,joint_min_C,joint_width,joint_params,joint_error,NFE_saved,joint_over_baseline_NFE
0,0.06,NaN,NaN,100,32,20259,0.059991,NaN,NaN
1,0.07,NaN,NaN,50,1,78,0.068389,NaN,NaN
2,0.08,500.0,0.070779,50,1,78,0.068389,450.0,0.10
3,0.09,500.0,0.070779,50,1,78,0.068389,450.0,0.10
4,0.10,500.0,0.070779,50,1,78,0.068389,450.0,0.10
5,0.12,500.0,0.070779,50,1,78,0.068389,450.0,0.10
6,0.15,200.0,0.144920,50,1,78,0.068389,150.0,0.25
7,0.20,10.0,0.199645,50,1,78,0.068389,-40.0,5.00



AUTOMATIC CHECKS
joint_policy_saves_NFE_somewhere: True
joint_policy_saves_2x_or_more_NFE_somewhere: True
C100_corrected_within_15pct_of_C500_corrected: False

Latency available: True

Saved outputs to: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\step6_joint_frontier_lean
